<a href="https://colab.research.google.com/github/aislam2346/aislam/blob/main/NYC_Taxi_Pipeline_Clean_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Section 1: Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ── Imports ──────────────────────────────────────────────────────────────────
import duckdb
import pandas as pd
import numpy as np
import requests
import os
import hashlib
import json
import gc
import holidays
from pathlib import Path
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ── Paths ─────────────────────────────────────────────────────────────────────
# Raw parquet files — where TLC files live on your Drive
PARQUET_DIR_2019 = Path("/content/drive/MyDrive/ODATA57078/nyc_taxi_data_2019")
PARQUET_DIR_2020 = Path("/content/drive/MyDrive/ODATA57078/nyc_taxi_data_2020")

# Support files
ZONE_LOOKUP_PATH = Path("/content/drive/MyDrive/ODATA57078/taxi_zone_lookup.csv")

# Output folders — created automatically if missing
BASE     = Path("/content/drive/MyDrive/ODATA57078")
GOLD_DIR = BASE / "gold"          # one parquet per month, e.g. gold/2019-01.parquet
FEAT_DIR = BASE / "features"      # final modeling table
CKPT_DIR = BASE / "checkpoints"   # watermark.json lives here
OUT_DIR  = BASE / "eda_outputs"   # saved plots

for d in [GOLD_DIR, FEAT_DIR, CKPT_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── DuckDB — lives in /tmp, NOT on Drive ─────────────────────────────────────
# This is the key change. /tmp is wiped when Colab disconnects, which is fine
# because your gold parquet files on Drive are the real persistent storage.
DB_PATH = "/tmp/taxi_work.duckdb"

def get_connection():
    con = duckdb.connect(DB_PATH)
    con.execute("PRAGMA threads=4")
    con.execute("PRAGMA memory_limit='3GB'")
    return con

# ── Months to process ────────────────────────────────────────────────────────
# Add 2020 months here when you're ready to ingest them.
# The watermark means already-completed months are always skipped.
MONTHS_TO_INGEST = [
    '2019-01', '2019-02', '2019-03',
    '2019-04', '2019-05', '2019-06',
    '2019-07', '2019-08', '2019-09',
    '2019-10', '2019-11', '2019-12',
    # '2020-01', '2020-02', '2020-03',   # uncomment when ready
    # '2020-04', '2020-05', '2020-06',
    # '2020-07', '2020-08', '2020-09',
    # '2020-10', '2020-11', '2020-12',
]

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 130, 'axes.titlesize': 11, 'axes.labelsize': 9,
    'figure.facecolor': '#0f0f1a', 'axes.facecolor': '#1a1a2e',
    'text.color': 'white', 'axes.labelcolor': 'white',
    'xtick.color': '#aaaaaa', 'ytick.color': '#aaaaaa',
})
COLORS = {
    'primary': '#4C9BE8', 'secondary': '#F5A623',
    'green': '#50C878',   'purple': '#A78BFA', 'red': '#E85D75',
}

print("Section 1 complete — paths and imports ready")
print(f"   GOLD_DIR : {GOLD_DIR}")
print(f"   DB_PATH  : {DB_PATH}  (local /tmp — not on Drive)")

Mounted at /content/drive
Section 1 complete — paths and imports ready
   GOLD_DIR : /content/drive/MyDrive/ODATA57078/gold
   DB_PATH  : /tmp/taxi_work.duckdb  (local /tmp — not on Drive)


## Section 2: Define Schema and Watermark

In [2]:
# ── Schema bootstrap ──────────────────────────────────────────────────────────
# raw_taxi_trips and staging_taxi_trips are TEMP tables — they exist only
# for the duration of processing one month, then disappear automatically.
# gold_hourly_demand and weather_hourly are permanent in /tmp DuckDB,
# but their durable copy is the parquet files on Drive.

def bootstrap_schema(con):
    """Create all tables. Safe to call every session — idempotent."""

    # Watermark — tracks what has been ingested (permanent, small)
    con.execute("""
        CREATE TABLE IF NOT EXISTS pipeline_watermark (
            file_name   VARCHAR PRIMARY KEY,
            file_month  VARCHAR,
            checksum    VARCHAR,
            row_count   BIGINT,
            ingested_at TIMESTAMP,
            layer       VARCHAR
        )
    """)

    # Raw — TEMP: holds one month of raw parquet data during processing
    # Dropped automatically when the connection closes
    con.execute("""
        CREATE TEMP TABLE IF NOT EXISTS raw_taxi_trips (
            VendorID              INTEGER,
            tpep_pickup_datetime  TIMESTAMP,
            tpep_dropoff_datetime TIMESTAMP,
            passenger_count       DOUBLE,
            trip_distance         DOUBLE,
            RatecodeID            DOUBLE,
            store_and_fwd_flag    VARCHAR,
            PULocationID          INTEGER,
            DOLocationID          INTEGER,
            payment_type          BIGINT,
            fare_amount           DOUBLE,
            extra                 DOUBLE,
            mta_tax               DOUBLE,
            tip_amount            DOUBLE,
            tolls_amount          DOUBLE,
            improvement_surcharge DOUBLE,
            total_amount          DOUBLE,
            congestion_surcharge  DOUBLE,
            airport_fee           DOUBLE,
            file_name             VARCHAR,
            file_month            VARCHAR,
            ingested_at           TIMESTAMP
        )
    """)

    # Staging — TEMP: holds one month of cleaned data during processing
    con.execute("""
        CREATE TEMP TABLE IF NOT EXISTS staging_taxi_trips (
            VendorID              INTEGER,
            tpep_pickup_datetime  TIMESTAMP,
            tpep_dropoff_datetime TIMESTAMP,
            passenger_count       INTEGER,
            trip_distance         DOUBLE,
            PULocationID          INTEGER,
            DOLocationID          INTEGER,
            payment_type          INTEGER,
            fare_amount           DOUBLE,
            tip_amount            DOUBLE,
            total_amount          DOUBLE,
            congestion_surcharge  DOUBLE,
            trip_duration_mins    DOUBLE,
            pickup_hour           INTEGER,
            pickup_dow            INTEGER,
            pickup_month          INTEGER,
            is_weekend            BOOLEAN,
            time_of_day           VARCHAR,
            file_month            VARCHAR,
            ingested_at           TIMESTAMP,
            rowhash               VARCHAR
        )
    """)

    # Gold — permanent in DuckDB session, but real persistence is parquet on Drive
    con.execute("""
        CREATE TABLE IF NOT EXISTS gold_hourly_demand (
            pickup_hour_ts      TIMESTAMP,
            file_month          VARCHAR,
            PULocationID        INTEGER,
            borough             VARCHAR,
            zone                VARCHAR,
            trip_count          BIGINT,
            avg_trip_distance   DOUBLE,
            avg_fare_amount     DOUBLE,
            avg_duration_mins   DOUBLE,
            avg_passenger_count DOUBLE,
            hour_of_day         INTEGER,
            day_of_week         INTEGER,
            is_weekend          BOOLEAN,
            is_holiday          BOOLEAN,
            temperature_2m      DOUBLE,
            precipitation       DOUBLE,
            snowfall            DOUBLE,
            windspeed_10m       DOUBLE,
            weathercode         INTEGER,
            is_snowing          BOOLEAN,
            is_raining          BOOLEAN,
            is_extreme_weather  BOOLEAN
        )
    """)

    # Weather — permanent in DuckDB session (fetched once from API)
    con.execute("""
        CREATE TABLE IF NOT EXISTS weather_hourly (
            weather_ts         TIMESTAMP PRIMARY KEY,
            temperature_2m     DOUBLE,
            precipitation      DOUBLE,
            snowfall           DOUBLE,
            windspeed_10m      DOUBLE,
            weathercode        INTEGER,
            is_snowing         BOOLEAN,
            is_raining         BOOLEAN,
            is_extreme_weather BOOLEAN
        )
    """)

    print("  Schema bootstrapped (idempotent)")


In [3]:
# ── Watermark — JSON on Drive, survives session restarts ──────────────────────
# This replaces the pipeline_watermark DuckDB table for session persistence.
# The DuckDB watermark table above is kept as a within-session fast lookup.
# The JSON file on Drive is the source of truth across sessions.

WATERMARK_PATH = CKPT_DIR / "watermark.json"

def _load_watermark_json():
    """Load watermark from Drive JSON. Returns empty dict on first run."""
    if WATERMARK_PATH.exists():
        return json.loads(WATERMARK_PATH.read_text())
    return {}

def _save_watermark_json(wm: dict):
    """Persist watermark dict back to Drive JSON."""
    WATERMARK_PATH.write_text(json.dumps(wm, indent=2, default=str))

def file_checksum(path):
    """MD5 checksum of a file — detects if source data changed."""
    md5 = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            md5.update(chunk)
    return md5.hexdigest()

def is_already_ingested(con, file_name, checksum, layer):
    """
    Check Drive JSON watermark first (persists across sessions),
    then DuckDB watermark (fast within-session lookup).
    Returns True only if file_name + checksum + layer all match.
    """
    wm = _load_watermark_json()
    key = f"{layer}::{file_name}"
    if key in wm:
        if wm[key]['checksum'] == checksum:
            return True
        else:
            print(f"  Checksum changed for {file_name} — re-ingesting")
            return False
    # Fallback: check DuckDB watermark table
    result = con.execute(
        "SELECT checksum FROM pipeline_watermark WHERE file_name = ? AND layer = ?",
        [file_name, layer]
    ).fetchone()
    if result and result[0] == checksum:
        return True
    if result and result[0] != checksum:
        print(f"  Checksum changed for {file_name} — re-ingesting")
    return False

def mark_ingested(con, file_name, file_month, checksum, row_count, layer):
    """
    Write to both Drive JSON (persistent) and DuckDB table (fast lookup).
    """
    # Drive JSON — survives session restarts
    wm = _load_watermark_json()
    key = f"{layer}::{file_name}"
    wm[key] = {
        'file_name':   file_name,
        'file_month':  file_month,
        'checksum':    checksum,
        'row_count':   row_count,
        'ingested_at': str(datetime.now()),
        'layer':       layer,
    }
    _save_watermark_json(wm)

    # DuckDB table — fast within-session lookup
    con.execute("""
        INSERT INTO pipeline_watermark
            (file_name, file_month, checksum, row_count, ingested_at, layer)
        VALUES (?, ?, ?, ?, now(), ?)
        ON CONFLICT (file_name) DO UPDATE SET
            checksum    = excluded.checksum,
            row_count   = excluded.row_count,
            ingested_at = excluded.ingested_at,
            layer       = excluded.layer
    """, [file_name, file_month, checksum, row_count, layer])

def show_watermark():
    """Print all ingested files — useful for debugging and audit."""
    wm = _load_watermark_json()
    if not wm:
        print("  No watermark entries yet")
        return
    df = pd.DataFrame(wm.values()).sort_values(['layer', 'file_month'])
    display(df[['layer', 'file_month', 'file_name', 'row_count', 'ingested_at']])

print("Section 2 complete — schema and watermark functions ready")

Section 2 complete — schema and watermark functions ready


## Section 3: Define Ingestion Functions

In [4]:
# ── Helper: find raw parquet for a given month ────────────────────────────────
def find_parquet(file_month):
    """
    Returns the parquet Path for a given month string like '2019-01'.
    Looks in 2019 folder or 2020 folder automatically.
    """
    year = file_month.split('-')[0]
    folder = PARQUET_DIR_2019 if year == '2019' else PARQUET_DIR_2020
    path = folder / f"yellow_tripdata_{file_month}.parquet"
    return path


# ── Layer 1: Raw ingestion ────────────────────────────────────────────────────
def ingest_raw(con, parquet_path, file_month):
    """
    Load one month parquet as-is into raw_taxi_trips TEMP table.
    Skips if gold parquet already exists on Drive (fastest check).
    Skips if checksum matches watermark (file unchanged).
    """
    # Fastest skip: gold parquet already built for this month
    gold_path = GOLD_DIR / f"{file_month}.parquet"
    if gold_path.exists():
        print(f"  {file_month}: gold parquet exists — skipping raw ingest")
        return False   # False = skip staging and gold too

    file_name = parquet_path.name
    checksum  = file_checksum(parquet_path)

    if is_already_ingested(con, file_name, checksum, layer='raw'):
        print(f"  Raw already ingested: {file_name}")
        return True    # True = proceed to staging

    # Clear any previous partial load for this month
    con.execute("DELETE FROM raw_taxi_trips WHERE file_month = ?", [file_month])

    con.execute(f"""
        INSERT INTO raw_taxi_trips
        SELECT
            VendorID, tpep_pickup_datetime, tpep_dropoff_datetime,
            passenger_count, trip_distance, RatecodeID, store_and_fwd_flag,
            PULocationID, DOLocationID, payment_type, fare_amount, extra,
            mta_tax, tip_amount, tolls_amount, improvement_surcharge,
            total_amount,
            TRY_CAST(congestion_surcharge AS DOUBLE),
            TRY_CAST(airport_fee          AS DOUBLE),
            '{file_name}'  AS file_name,
            '{file_month}' AS file_month,
            now()          AS ingested_at
        FROM read_parquet('{parquet_path}')
    """)

    row_count = con.execute(
        "SELECT COUNT(*) FROM raw_taxi_trips WHERE file_month = ?", [file_month]
    ).fetchone()[0]

    mark_ingested(con, file_name, file_month, checksum, row_count, layer='raw')
    print(f"  Raw  | {file_name} | {row_count:,} rows")
    return True


In [5]:
# ── Layer 2: Staging ingestion ────────────────────────────────────────────────
def ingest_staging(con, file_month):
    """
    Promote one month raw → staging with cleaning rules:
      R1: Valid timestamps (non-null, pickup < dropoff, correct year)
      R2: Passenger count 1–6
      R3: Fare >= $2.50, total_amount > 0
      R4: Trip distance 0.1–50 miles
      R5: Trip duration 1–150 minutes
    Also computes rowhash for duplicate detection.
    """
    expected_year = int(file_month.split('-')[0])

    # Pre-cleaning audit — shows how many rows fail each rule
    df_audit = con.execute(f"""
        SELECT
            file_month,
            COUNT(*) AS total_rows,
            SUM(CASE WHEN tpep_pickup_datetime IS NULL
                      OR tpep_dropoff_datetime IS NULL
                      OR tpep_pickup_datetime >= tpep_dropoff_datetime
                      OR YEAR(tpep_pickup_datetime)  != {expected_year}
                      OR YEAR(tpep_dropoff_datetime) != {expected_year}
                                                         THEN 1 ELSE 0 END) AS bad_timestamps,
            SUM(CASE WHEN passenger_count IS NULL
                      OR passenger_count < 1
                      OR passenger_count > 6             THEN 1 ELSE 0 END) AS bad_passengers,
            SUM(CASE WHEN fare_amount IS NULL
                      OR fare_amount < 2.50
                      OR total_amount IS NULL
                      OR total_amount <= 0               THEN 1 ELSE 0 END) AS bad_fares,
            SUM(CASE WHEN trip_distance <= 0.1
                      OR trip_distance > 50              THEN 1 ELSE 0 END) AS bad_distance
        FROM raw_taxi_trips
        WHERE file_month = '{file_month}'
        GROUP BY file_month
    """).fetchdf()
    print(f"\n  Pre-cleaning audit for {file_month}:")
    display(df_audit)

    con.execute("DELETE FROM staging_taxi_trips WHERE file_month = ?", [file_month])

    con.execute(f"""
        INSERT INTO staging_taxi_trips
        SELECT DISTINCT
            VendorID, tpep_pickup_datetime, tpep_dropoff_datetime,
            CAST(passenger_count AS INTEGER),
            trip_distance,
            PULocationID, DOLocationID,
            CAST(payment_type AS INTEGER),
            fare_amount, tip_amount, total_amount, congestion_surcharge,
            DATEDIFF('minute', tpep_pickup_datetime, tpep_dropoff_datetime) AS trip_duration_mins,
            HOUR(tpep_pickup_datetime)                        AS pickup_hour,
            DAYOFWEEK(tpep_pickup_datetime)                   AS pickup_dow,
            MONTH(tpep_pickup_datetime)                       AS pickup_month,
            DAYOFWEEK(tpep_pickup_datetime) IN (0, 6)        AS is_weekend,
            CASE
                WHEN HOUR(tpep_pickup_datetime) BETWEEN 6  AND 11 THEN 'Morning'
                WHEN HOUR(tpep_pickup_datetime) BETWEEN 12 AND 16 THEN 'Afternoon'
                WHEN HOUR(tpep_pickup_datetime) BETWEEN 17 AND 21 THEN 'Evening'
                ELSE 'Night'
            END                                               AS time_of_day,
            file_month,
            ingested_at,
            md5(
                COALESCE(CAST(VendorID               AS VARCHAR), '') ||
                COALESCE(CAST(tpep_pickup_datetime   AS VARCHAR), '') ||
                COALESCE(CAST(tpep_dropoff_datetime  AS VARCHAR), '') ||
                COALESCE(CAST(PULocationID           AS VARCHAR), '') ||
                COALESCE(CAST(DOLocationID           AS VARCHAR), '') ||
                COALESCE(CAST(trip_distance          AS VARCHAR), '') ||
                COALESCE(CAST(fare_amount            AS VARCHAR), '') ||
                COALESCE(CAST(total_amount           AS VARCHAR), '') ||
                COALESCE(CAST(passenger_count        AS VARCHAR), '')
            ) AS rowhash
        FROM raw_taxi_trips
        WHERE file_month = '{file_month}'
          AND tpep_pickup_datetime  IS NOT NULL
          AND tpep_dropoff_datetime IS NOT NULL
          AND tpep_pickup_datetime  < tpep_dropoff_datetime
          AND YEAR(tpep_pickup_datetime)  = {expected_year}
          AND YEAR(tpep_dropoff_datetime) = {expected_year}
          AND passenger_count IS NOT NULL
          AND passenger_count BETWEEN 1 AND 6
          AND fare_amount  IS NOT NULL
          AND fare_amount  >= 2.50
          AND total_amount IS NOT NULL
          AND total_amount >  0
          AND trip_distance BETWEEN 0.1 AND 50
          AND DATEDIFF('minute', tpep_pickup_datetime, tpep_dropoff_datetime) BETWEEN 1 AND 150
    """)

    # Post-cleaning sense check
    df_sense = con.execute(f"""
        SELECT
            file_month,
            COUNT(*) AS staging_rows,
            (COUNT(rowhash) - COUNT(DISTINCT rowhash)) AS duplicate_hashes,
            SUM(CASE WHEN fare_amount < 2.50 THEN 1 ELSE 0 END) AS bad_fare_remaining,
            SUM(CASE WHEN passenger_count NOT BETWEEN 1 AND 6
                                              THEN 1 ELSE 0 END) AS bad_pax_remaining,
            SUM(CASE WHEN total_amount <= 0   THEN 1 ELSE 0 END) AS bad_total_remaining,
            SUM(CASE WHEN trip_duration_mins NOT BETWEEN 1 AND 150
                                              THEN 1 ELSE 0 END) AS bad_duration_remaining
        FROM staging_taxi_trips
        WHERE file_month = '{file_month}'
        GROUP BY file_month
    """).fetchdf()
    print(f"\n  Post-cleaning sense check for {file_month}:")
    display(df_sense)

    # Show up to 5 duplicate hashes for inspection
    df_dups = con.execute(f"""
        WITH dup_list AS (
            SELECT rowhash
            FROM staging_taxi_trips
            WHERE file_month = '{file_month}'
            GROUP BY rowhash HAVING COUNT(*) > 1
            LIMIT 5
        )
        SELECT * FROM staging_taxi_trips
        WHERE rowhash IN (SELECT rowhash FROM dup_list)
        ORDER BY rowhash
    """).fetchdf()
    if not df_dups.empty:
        print(f"  Duplicate hash sample for {file_month}:")
        display(df_dups)

    row_count = int(df_sense['staging_rows'].iloc[0]) if not df_sense.empty else 0
    print(f"  Staging | {file_month} | {row_count:,} rows")

In [6]:
# ── Layer 3: Gold ingestion ───────────────────────────────────────────────────
def build_gold(con, file_month, zone_lookup_path=None):
    """
    Aggregate staging → gold at hourly grain per pickup zone.
    Joins weather_hourly (must be ingested first).
    Computes is_holiday in Python then passes to SQL (fixes all-False bug).
    Exports gold to Drive parquet and marks watermark.
    Grain: (pickup_hour_ts, PULocationID)
    """
    # Register zone lookup if not already loaded
    try:
        con.execute("SELECT 1 FROM taxi_zones LIMIT 1")
    except Exception:
        src = zone_lookup_path or str(ZONE_LOOKUP_PATH)
        df_zones = pd.read_csv(src)
        df_zones.columns = df_zones.columns.str.lower().str.strip()
        con.register('taxi_zones', df_zones)
        print(f"  Zone lookup registered: {len(df_zones)} zones")

    # Check weather data exists for this month
    weather_check = con.execute(f"""
        SELECT COUNT(*) FROM weather_hourly
        WHERE weather_ts BETWEEN '{file_month}-01'
                             AND '{file_month}-01'::DATE + INTERVAL 1 MONTH
    """).fetchone()[0]
    if weather_check == 0:
        print(f"  No weather data for {file_month} — run ingest_weather() first")
        return

    # ── is_holiday: computed in Python, passed to SQL as a temp table ─────────
    # This is the fix for the all-False bug. The SQL IN (...) approach with
    # a tuple of strings failed silently. Registering a DataFrame works reliably.
    nyc_holidays = holidays.US(state='NY', years=[2019, 2020, 2021])
    df_holidays = pd.DataFrame({
        'holiday_date': [str(d) for d in nyc_holidays.keys()]
    })
    con.register('temp_holidays', df_holidays)

    con.execute("DELETE FROM gold_hourly_demand WHERE file_month = ?", [file_month])

    con.execute(f"""
        INSERT INTO gold_hourly_demand
        SELECT
            DATE_TRUNC('hour', s.tpep_pickup_datetime)              AS pickup_hour_ts,
            s.file_month,
            s.PULocationID,
            z.borough,
            z.zone,
            COUNT(*)                                                AS trip_count,
            ROUND(AVG(s.trip_distance),      4)                     AS avg_trip_distance,
            ROUND(AVG(s.fare_amount),        4)                     AS avg_fare_amount,
            ROUND(AVG(s.trip_duration_mins), 4)                     AS avg_duration_mins,
            ROUND(AVG(s.passenger_count),    4)                     AS avg_passenger_count,
            HOUR(s.tpep_pickup_datetime)                            AS hour_of_day,
            DAYOFWEEK(s.tpep_pickup_datetime)                       AS day_of_week,
            DAYOFWEEK(s.tpep_pickup_datetime) IN (0, 6)             AS is_weekend,
            CASE WHEN CAST(DATE_TRUNC('hour', s.tpep_pickup_datetime) AS DATE)::VARCHAR
                      IN (SELECT holiday_date FROM temp_holidays)
                 THEN TRUE ELSE FALSE END                           AS is_holiday,
            w.temperature_2m, w.precipitation, w.snowfall,
            w.windspeed_10m, w.weathercode,
            w.is_snowing, w.is_raining, w.is_extreme_weather
        FROM staging_taxi_trips s
        LEFT JOIN taxi_zones z    ON s.PULocationID = z.locationid
        LEFT JOIN weather_hourly w
               ON DATE_TRUNC('hour', s.tpep_pickup_datetime) = w.weather_ts
        WHERE s.file_month = '{file_month}'
          AND z.borough NOT IN ('EWR', 'Unknown')
        GROUP BY
            DATE_TRUNC('hour', s.tpep_pickup_datetime),
            s.file_month, s.PULocationID, z.borough, z.zone,
            HOUR(s.tpep_pickup_datetime),
            DAYOFWEEK(s.tpep_pickup_datetime),
            DAYOFWEEK(s.tpep_pickup_datetime) IN (0, 6),
            w.temperature_2m, w.precipitation, w.snowfall,
            w.windspeed_10m, w.weathercode,
            w.is_snowing, w.is_raining, w.is_extreme_weather
    """)

    row_count = con.execute(
        "SELECT COUNT(*) FROM gold_hourly_demand WHERE file_month = ?", [file_month]
    ).fetchone()[0]
    print(f"  Gold | {file_month} | {row_count:,} hourly zone records")

    # ── Export to Drive parquet — the persistent copy ─────────────────────────
    gold_path = GOLD_DIR / f"{file_month}.parquet"
    con.execute(f"""
        COPY (SELECT * FROM gold_hourly_demand WHERE file_month = '{file_month}')
        TO '{gold_path}' (FORMAT PARQUET)
    """)
    print(f"  Saved: {gold_path}")

    # Mark gold as done in watermark (checksum = 'derived' since it's computed)
    file_name = f"yellow_tripdata_{file_month}.parquet"
    mark_ingested(con, file_name, file_month,
                  checksum='derived', row_count=row_count, layer='gold')

print(" Section 3 complete — ingestion functions ready")


 Section 3 complete — ingestion functions ready


## Section 4: Weather Ingestion

In [7]:
def ingest_weather(con, start_date='2019-01-01', end_date='2020-12-31'):
    """s
    Fetch hourly weather from Open-Meteo archive API for NYC.
    Watermark-based — resumes from last loaded hour if interrupted.

    Units: temperature_2m=°F, precipitation=mm, snowfall=cm, windspeed=km/h
    Extreme thresholds: snowfall > 7.62 cm OR precipitation > 25.4 mm
    """
    last_ts = con.execute("SELECT MAX(weather_ts) FROM weather_hourly").fetchone()[0]

    if last_ts is not None:
        fetch_from = str((last_ts + timedelta(hours=1)).date())
        if fetch_from > end_date:
            total = con.execute("SELECT COUNT(*) FROM weather_hourly").fetchone()[0]
            print(f"  Weather already loaded to {last_ts} ({total:,} hours)")
            return
        print(f"  Resuming weather from {fetch_from} → {end_date}")
    else:
        fetch_from = start_date
        print(f"  Fetching hourly weather {fetch_from} → {end_date}")

    resp = requests.get(
        'https://archive-api.open-meteo.com/v1/archive',
        params={
            'latitude': 40.7128, 'longitude': -74.0060,
            'start_date': fetch_from, 'end_date': end_date,
            'hourly': [
                'temperature_2m', 'precipitation', 'snowfall',
                'windspeed_10m', 'weathercode'
            ],
            'timezone': 'America/New_York',
        },
        timeout=60,
    )
    resp.raise_for_status()
    data = resp.json()

    df_w = pd.DataFrame({
        'weather_ts':    pd.to_datetime(data['hourly']['time']),
        'temperature_2m': data['hourly']['temperature_2m'],
        'precipitation':  data['hourly']['precipitation'],
        'snowfall':       data['hourly']['snowfall'],
        'windspeed_10m':  data['hourly']['windspeed_10m'],
        'weathercode':    data['hourly']['weathercode'],
    })

    df_w['is_snowing']         = df_w['snowfall']       > 7.62
    df_w['is_raining']         = df_w['precipitation']  > 0
    df_w['is_extreme_weather'] = (df_w['snowfall'] > 7.62) | (df_w['precipitation'] > 25.4)

    con.register('weather_staging', df_w)
    con.execute("""
        INSERT OR REPLACE INTO weather_hourly
        SELECT * FROM weather_staging
    """)
    con.unregister('weather_staging')

    total = con.execute("SELECT COUNT(*) FROM weather_hourly").fetchone()[0]
    print(f"  Weather loaded: {total:,} hours ({fetch_from} → {end_date})")

print(" Section 4 complete — weather function ready")

 Section 4 complete — weather function ready


## Section 5: Main Pipeline Loop

In [8]:
con = get_connection()
bootstrap_schema(con)

# Fetch weather first — covers all months in one API call
ingest_weather(con, start_date='2019-01-01', end_date='2020-12-31')

PARQUET_DRIVE = PARQUET_DIR_2019   # change to PARQUET_DIR_2020 for 2020 months

for month in MONTHS_TO_INGEST:
    print(f"\n{'='*55}")
    print(f"  Processing: {month}")
    print(f"{'='*55}")

    parquet_path = PARQUET_DRIVE / f"yellow_tripdata_{month}.parquet"

    if not parquet_path.exists():
        print(f"  File not found: {parquet_path.name} — skipping")
        continue

    did_ingest = ingest_raw(con, parquet_path, month)

    if not did_ingest:
        # Gold already exists for this month — nothing to do
        continue

    ingest_staging(con, month)
    build_gold(con, file_month=month, zone_lookup_path=str(ZONE_LOOKUP_PATH))

    # ── Free memory: clear this month from raw + staging ─────────────────────
    # Gold is now safely in parquet on Drive — no need to keep these in DuckDB
    con.execute("DELETE FROM raw_taxi_trips    WHERE file_month = ?", [month])
    con.execute("DELETE FROM staging_taxi_trips WHERE file_month = ?", [month])
    con.execute("CHECKPOINT")
    gc.collect()
    print(f"  Memory cleared for {month}")

print("\n Pipeline complete")
print(f"   DuckDB size: {os.path.getsize(DB_PATH)/1024/1024:.1f} MB  (local /tmp)")

# Optional: view the full watermark log
show_watermark()

  Schema bootstrapped (idempotent)
  Fetching hourly weather 2019-01-01 → 2020-12-31
  Weather loaded: 17,544 hours (2019-01-01 → 2020-12-31)

  Processing: 2019-01


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Raw  | yellow_tripdata_2019-01.parquet | 7,696,617 rows

  Pre-cleaning audit for 2019-01:


,file_month,total_rows,bad_timestamps,bad_passengers,bad_fares,bad_distance
0,2019-01,7696617,6997.0,146110.0,10879.0,79210.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Staging | 2019-12 | 6,591,564 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Gold | 2019-12 | 81,335 hourly zone records
  Saved: /content/drive/MyDrive/ODATA57078/gold/2019-12.parquet
  Memory cleared for 2019-12

 Pipeline complete
   DuckDB size: 33.5 MB  (local /tmp)


,layer,file_month,file_name,row_count,ingested_at
1,gold,2019-01,yellow_tripdata_2019-01.parquet,98690,2026-04-04 05:25:04.790786
3,gold,2019-02,yellow_tripdata_2019-02.parquet,91415,2026-04-04 05:26:16.385341
5,gold,2019-03,yellow_tripdata_2019-03.parquet,101211,2026-04-04 05:27:24.007834
7,gold,2019-04,yellow_tripdata_2019-04.parquet,91595,2026-04-04 05:28:34.045072
9,gold,2019-05,yellow_tripdata_2019-05.parquet,92268,2026-04-04 05:29:50.383803
11,gold,2019-06,yellow_tripdata_2019-06.parquet,89807,2026-04-04 05:30:45.252924
13,gold,2019-07,yellow_tripdata_2019-07.parquet,88272,2026-04-04 05:31:41.308638
15,gold,2019-08,yellow_tripdata_2019-08.parquet,84939,2026-04-04 05:32:43.376305
17,gold,2019-09,yellow_tripdata_2019-09.parquet,82842,2026-04-04 05:33:46.039025
19,gold,2019-10,yellow_tripdata_2019-10.parquet,82785,2026-04-04 05:35:03.372177


## Section 6: Load Gold data and build Features for modelling

In [18]:
def load_gold(months):
    """
    Load gold parquet files from Drive for the given months.
    Returns a combined pandas DataFrame sorted by zone and timestamp.
    """
    frames = []
    for m in months:
        path = GOLD_DIR / f"{m}.parquet"
        if path.exists():
            frames.append(pd.read_parquet(path))
        else:
            print(f"  Warning: {m}.parquet not found \u2014 run pipeline first")
    if not frames:
        raise FileNotFoundError("No gold parquet files found. Run Section 5 first.")
    df = pd.concat(frames, ignore_index=True)
    df['pickup_hour_ts'] = pd.to_datetime(df['pickup_hour_ts'])
    df = df.sort_values(['PULocationID', 'pickup_hour_ts']).reset_index(drop=True)
    print(f"  Loaded {len(df):,} rows across {len(frames)} months")
    return df


def build_lag_features(df,
                        location_col='PULocationID',
                        datetime_col='pickup_hour_ts',
                        demand_col='trip_count'):
    """
    Adds lag and rolling features to the gold DataFrame.
    Must be called on the FULL dataset (all months together) so that
    lags across month boundaries are computed correctly.
    Missing hours per zone are filled with 0 trips (spine reindex).
    """
    # Check for and display duplicate entries before reindexing
    initial_rows = len(df)
    df_duplicates = df[df.duplicated(subset=[location_col, datetime_col], keep=False)]
    if not df_duplicates.empty:
        print(f"  Warning: Found {len(df_duplicates)} duplicate rows for {location_col} and {datetime_col}. Displaying first 10 duplicates:")
        display(df_duplicates.head(10))
        print(f"  Aggregating {len(df_duplicates)} duplicate rows by mean...")

        # Define aggregation dictionary
        agg_funcs = {}
        for col in df.columns:
            if col not in [location_col, datetime_col]:
                if pd.api.types.is_numeric_dtype(df[col]):
                    agg_funcs[col] = 'mean'
                else:
                    # For non-numeric, take the mode. Handle cases where mode might be empty (e.g., all NaNs).
                    agg_funcs[col] = lambda x: x.mode()[0] if not x.mode().empty else None

        df = df.groupby([location_col, datetime_col]).agg(agg_funcs).reset_index()
        print(f"  Aggregated to {len(df)} unique rows.")

    # Fill complete hourly spine per zone \u2014 prevents wrong lag offsets
    all_hours = pd.date_range(
        df[datetime_col].min(),
        df[datetime_col].max(),
        freq='h'
    )
    all_zones = df[location_col].unique()
    spine = pd.MultiIndex.from_product(
        [all_zones, all_hours], names=[location_col, datetime_col]
    )
    df = (df.set_index([location_col, datetime_col])
            .reindex(spine)
            .reset_index())
    df[demand_col] = df[demand_col].fillna(0)
    # Re-derive file_month after potential reindexing/aggregation to ensure consistency
    df['file_month'] = df[datetime_col].dt.to_period('M').astype(str)

    # Lag features \u2014 grouped by zone so lags don't bleed across zones
    grp = df.groupby(location_col)[demand_col]
    df['lag_1h']    = grp.shift(1)
    df['lag_2h']    = grp.shift(2)
    df['lag_24h']   = grp.shift(24)
    df['lag_168h']  = grp.shift(168)   # 1 week back

    # Rolling averages \u2014 shift(1) before rolling avoids data leakage
    df['roll_3h']   = grp.transform(lambda x: x.shift(1).rolling(3,   min_periods=1).mean())
    df['roll_24h']  = grp.transform(lambda x: x.shift(1).rolling(24,  min_periods=1).mean())
    df['roll_168h'] = grp.transform(lambda x: x.shift(1).rolling(168, min_periods=1).mean())

    print(f"  Lag features built \u2014 {len(df):,} rows (includes spine fill)")
    return df

Then build features

In [19]:
# ── Load and build features ───────────────────────────────────────────────────
TRAIN_MONTHS = [
    '2019-01', '2019-02', '2019-03', '2019-04', '2019-05', '2019-06',
    '2019-07', '2019-08', '2019-09',
]

df_gold = load_gold(TRAIN_MONTHS)
df_gold = build_lag_features(df_gold)

# Drop first 168h per zone — these have unavoidably NaN lags
df_gold = df_gold.dropna(subset=['lag_168h'])

print(f"\n  Final modelling rows: {len(df_gold):,}")
print(f"  Columns: {list(df_gold.columns)}")

# Optional: save features to Drive so you don't have to rebuild each session
feat_path = FEAT_DIR / "features_train.parquet"
df_gold.to_parquet(feat_path, index=False)
print(f"\n  Features saved: {feat_path}")

print(" Section 6 complete — df_gold ready for modelling")

  Loaded 821,039 rows across 9 months


,pickup_hour_ts,file_month,PULocationID,borough,zone,trip_count,avg_trip_distance,avg_fare_amount,avg_duration_mins,avg_passenger_count,...,is_weekend,is_holiday,temperature_2m,precipitation,snowfall,windspeed_10m,weathercode,is_snowing,is_raining,is_extreme_weather
1842,2019-01-31 23:00:00,2019-01,4,Manhattan,Alphabet City,40,2.4082,10.5875,11.0500,1.6000,...,False,False,-10.2,0.0,0.0,12.7,0,False,False,False
1843,2019-01-31 23:00:00,2019-02,4,Manhattan,Alphabet City,1,4.6600,15.5000,16.0000,1.0000,...,False,False,-10.2,0.0,0.0,12.7,0,False,False,False
2514,2019-02-28 23:00:00,2019-02,4,Manhattan,Alphabet City,43,2.6944,11.7209,13.0233,1.3721,...,False,False,-3.5,0.0,0.0,8.2,3,False,False,False
2515,2019-02-28 23:00:00,2019-03,4,Manhattan,Alphabet City,1,2.9300,14.0000,18.0000,1.0000,...,False,False,-3.5,0.0,0.0,8.2,3,False,False,False
3975,2019-05-01 00:00:00,2019-04,4,Manhattan,Alphabet City,1,0.5200,4.0000,2.0000,1.0000,...,False,False,9.8,0.0,0.0,6.6,1,False,False,False
3976,2019-05-01 00:00:00,2019-05,4,Manhattan,Alphabet City,7,3.1686,12.0714,11.7143,1.2857,...,False,False,9.8,0.0,0.0,6.6,1,False,False,False
4717,2019-05-31 23:00:00,2019-05,4,Manhattan,Alphabet City,36,2.7211,12.5000,14.9444,1.6389,...,False,False,18.3,0.0,0.0,7.0,0,False,False,False
4718,2019-05-31 23:00:00,2019-06,4,Manhattan,Alphabet City,2,2.3750,10.2500,10.0000,1.0000,...,False,False,18.3,0.0,0.0,7.0,0,False,False,False
6172,2019-07-31 23:00:00,2019-07,4,Manhattan,Alphabet City,17,2.2900,9.8824,10.1765,1.5294,...,False,False,23.6,0.0,0.0,3.4,3,False,False,False
6173,2019-07-31 23:00:00,2019-08,4,Manhattan,Alphabet City,3,7.0233,22.6667,20.0000,2.6667,...,False,False,23.6,0.0,0.0,3.4,3,False,False,False


  Aggregating 2780 duplicate rows by mean...
  Aggregated to 819649 unique rows.
  Lag features built — 2,275,260 rows (includes spine fill)

  Final modelling rows: 2,231,580
  Columns: ['PULocationID', 'pickup_hour_ts', 'file_month', 'borough', 'zone', 'trip_count', 'avg_trip_distance', 'avg_fare_amount', 'avg_duration_mins', 'avg_passenger_count', 'hour_of_day', 'day_of_week', 'is_weekend', 'is_holiday', 'temperature_2m', 'precipitation', 'snowfall', 'windspeed_10m', 'weathercode', 'is_snowing', 'is_raining', 'is_extreme_weather', 'lag_1h', 'lag_2h', 'lag_24h', 'lag_168h', 'roll_3h', 'roll_24h', 'roll_168h']

  Features saved: /content/drive/MyDrive/ODATA57078/features/features_train.parquet
 Section 6 complete — df_gold ready for modelling


In [20]:
display(df_gold)

,PULocationID,pickup_hour_ts,file_month,borough,zone,trip_count,avg_trip_distance,avg_fare_amount,avg_duration_mins,avg_passenger_count,...,is_snowing,is_raining,is_extreme_weather,lag_1h,lag_2h,lag_24h,lag_168h,roll_3h,roll_24h,roll_168h
168,2,2019-01-08 00:00:00,2019-01,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.000000,0.000000,0.017857
169,2,2019-01-08 01:00:00,2019-01,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.000000,0.000000,0.017857
170,2,2019-01-08 02:00:00,2019-01,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.000000,0.000000,0.017857
171,2,2019-01-08 03:00:00,2019-01,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.000000,0.000000,0.017857
172,2,2019-01-08 04:00:00,2019-01,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.000000,0.000000,0.017857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2275255,263,2019-12-31 10:00:00,2019-12,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
2275256,263,2019-12-31 11:00:00,2019-12,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
2275257,263,2019-12-31 12:00:00,2019-12,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
2275258,263,2019-12-31 13:00:00,2019-12,Manhattan,Yorkville West,1.0,1.92,9.0,10.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000


In [22]:
print("\nNaN values per column in df_gold (before dropping rows for lag_168h):\n")
# Load df_gold again to see NaNs *before* the final dropna in JBH7OkTq8DKe
# This is for diagnostic purposes, the actual df_gold used for modeling will have these dropped.
# df_gold_temp_for_nan_check = load_gold(TRAIN_MONTHS)
# df_gold_temp_for_nan_check = build_lag_features(df_gold_temp_for_nan_check)

display(df_gold.isnull().sum().sort_values(ascending=False))


NaN values per column in df_gold (before dropping rows for lag_168h):



,0
zone,1434358
borough,1434358
avg_duration_mins,1434358
avg_fare_amount,1434358
avg_trip_distance,1434358
day_of_week,1434358
hour_of_day,1434358
is_weekend,1434358
avg_passenger_count,1434358
snowfall,1434358


## Section 7: Baseline Modelling

In [23]:
FEATURES = [
    'hour_of_day', 'day_of_week', 'is_weekend', 'month_of_year',
    'PULocationID',
    'temperature_2m', 'precipitation', 'snowfall', 'windspeed_10m',
    'is_snowing', 'is_raining', 'is_extreme_weather',
    'is_holiday',
    'lag_1h', 'lag_2h', 'lag_24h', 'lag_168h',
    'roll_3h', 'roll_24h', 'roll_168h',
]
TARGET = 'trip_count'

# Chronological split — never use random split for time series
df_model = df_gold.sort_values('pickup_hour_ts').reset_index(drop=True)

# Add month_of_year if not already present
if 'month_of_year' not in df_model.columns:
    df_model['month_of_year'] = df_model['pickup_hour_ts'].dt.month

TRAIN_END = '2019-07-31 23:00:00'
VAL_START = '2019-08-01 00:00:00'

train = df_model[df_model['pickup_hour_ts'] <= TRAIN_END]
val   = df_model[df_model['pickup_hour_ts'] >= VAL_START]

X_train, y_train = train[FEATURES], train[TARGET]
X_val,   y_val   = val[FEATURES],   val[TARGET]

print(f"Train: {len(X_train):,} rows  ({train['pickup_hour_ts'].min().date()} → {train['pickup_hour_ts'].max().date()})")
print(f"Val:   {len(X_val):,}   rows  ({val['pickup_hour_ts'].min().date()} → {val['pickup_hour_ts'].max().date()})")

params = {
    'objective':          'regression',
    'metric':             'mae',
    'n_estimators':       2000,
    'learning_rate':      0.05,
    'num_leaves':         63,
    'min_child_samples':  20,
    'subsample':          0.8,
    'colsample_bytree':   0.8,
    'random_state':       42,
    'verbose':            -1,
}

model = lgb.LGBMRegressor(**params)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100),
    ]
)

y_pred = np.clip(model.predict(X_val), 0, None)
mae    = mean_absolute_error(y_val, y_pred)
rmse   = np.sqrt(mean_squared_error(y_val, y_pred))
nmae   = mae / y_val.mean()

print(f"\n── Baseline Results ──────────────────────────────────")
print(f"  MAE:            {mae:.3f} trips")
print(f"  RMSE:           {rmse:.3f} trips")
print(f"  Normalised MAE: {nmae:.3%}")
print(f"  Mean demand:    {y_val.mean():.2f} trips/hr")
print(f"  Best iteration: {model.best_iteration_}")

print(" Section 7 complete — baseline model trained")


Train: 1,279,200 rows  (2019-01-08 → 2019-07-31)
Val:   952,380   rows  (2019-08-01 → 2019-12-31)
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l1: 2.391
[200]	valid_0's l1: 2.06512
[300]	valid_0's l1: 2.0173
[400]	valid_0's l1: 1.96021
[500]	valid_0's l1: 1.9053
[600]	valid_0's l1: 1.87493
[700]	valid_0's l1: 1.82308
[800]	valid_0's l1: 1.78873
[900]	valid_0's l1: 1.77099
[1000]	valid_0's l1: 1.75647
[1100]	valid_0's l1: 1.74608
[1200]	valid_0's l1: 1.73181
[1300]	valid_0's l1: 1.71465
[1400]	valid_0's l1: 1.6991
[1500]	valid_0's l1: 1.69048
[1600]	valid_0's l1: 1.68357
[1700]	valid_0's l1: 1.6782
[1800]	valid_0's l1: 1.6705
[1900]	valid_0's l1: 1.65515
[2000]	valid_0's l1: 1.64675
Did not meet early stopping. Best iteration is:
[1981]	valid_0's l1: 1.64521

── Baseline Results ──────────────────────────────────
  MAE:            1.622 trips
  RMSE:           8.362 trips
  Normalised MAE: 12.881%
  Mean demand:    12.59 trips/hr
  Best iteration: 1981
 S